# HTR with Claude

In [27]:
import anthropic
from dotenv import load_dotenv
import os
from pathlib import Path
import regex

In [3]:
load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [48]:
prompt = """Dear Claude, here is an image displaying two pages with Dutch text 
related to an inheritance. I am interested in the information in the text block 
on the top right of the right page. Could check if that part contains a text like: 
"Memorie van aangifte der nalatenschap van"? If that is the case, can you give me 
the information which follows next? This is 1. the name of the deceased, 2. the 
place of death, and 3. the date of death. Both place and date could be missing. 
If you see a big number next to the text block, that is the act number, which is 
interesting as well. Please return this all information in well-formatted JSON 
format with the keys "act", "name", "place" and "date" and without any comments. 
If the top right of the right page contains a different text or no text at all, 
please return an empty JSON structure."""

In [52]:
base_directory = "../memories_crawl/scans/bhic/Eindhoven/deel_84"
starting_pages = [x for x in range(1, 11)]
results = []
for file_name in sorted(os.listdir(base_directory)):
    try:
        page_number = int(regex.sub("^0+", "", file_name.split("_")[-1].split('.')[0]))
    except ValueError:
        continue
    if page_number in starting_pages:
        try:
            upload_response = client.beta.files.upload(file=Path(os.path.join(base_directory, file_name)))
            message = client.beta.messages.create(
                model="claude-sonnet-4-6",
                max_tokens=1024,
                messages=[
                     {"role": "user", 
                      "content": [
                        {"type": "image",
                         "source": {"type": "file",
                                    "file_id": upload_response.id
                                   }
                        },
                        {"type": "text", "text": prompt}
                     ]}
                ],
                betas=["files-api-2025-04-14"]
            )
            results.append(message.content[0].text)
        finally:
            client.beta.files.delete(upload_response.id)

In [54]:
for counter, result in enumerate(results):
    print(f"{counter + 1:2d}. {result}")

 1. ```json
{}
```
 2. ```json
{
  "act": "1",
  "name": "Dorothea van Ummelen",
  "place": "Aalst",
  "date": "10 januari 1889"
}
```
 3. ```json
{}
```
 4. ```json
{
  "act": "2",
  "name": "Johanna Maria Davis weduwe van Francis Boon",
  "place": "Bergen op Zoom",
  "date": "1 january 1879"
}
```
 5. ```json
{
  "act": "3",
  "name": "Wouter van Lieshout",
  "place": "Bergeyk",
  "date": "16 January 1819"
}
```
 6. ```json
{
  "act": "3",
  "name": "Wouter van Lieshout",
  "place": "Bergeijk",
  "date": "16 January 1819"
}
```
 7. ```json
{
  "act": "4",
  "name": "Christina de Krom weduwe Johannes Sanders",
  "place": "Bergijk",
  "date": "23 januari 1869"
}
```
 8. ```json
{
  "act": "5",
  "name": "Jacobus Vormisje",
  "place": null,
  "date": "1809-01-08"
}
```
 9. ```json
{
  "act": "3/7197",
  "name": "Lambertus Versmissen",
  "place": "Eersel",
  "date": "12 July 1889"
}
```
10. ```json
{}
```

The top right of the right page does not contain a "Memorie van aangifte der nalat